In [1]:
import os
import re
import json
import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

# Ensure consistent working directory and output folders
WORK_DIR = os.path.expanduser("~/LB2_Project")
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

def get_kingdom(entry):
    lineage = entry.get("organism", {}).get("lineage", [])
    if "Metazoa" in lineage:
        return "Metazoa"
    elif "Viridiplantae" in lineage:
        return "Viridiplantae"
    elif "Fungi" in lineage:
        return "Fungi"
    else:
        return "Other"

def filter_entry_positive(entry):
    try:
        e_pos = int(entry["features"][0]["location"]["end"]["value"])
        assert(entry["features"][0]["description"] == "")
        assert(e_pos > 13)
    except:
        return False
    return True

def filter_entry_negative(entry):
    return True

def json_to_tsv_positive(entry):
    return (
        entry["primaryAccession"],
        entry["organism"]["scientificName"],
        get_kingdom(entry),
        entry["sequence"]["length"],
        entry["features"][0]["location"]["end"]["value"]
    )

def json_to_tsv_negative(entry):
    tm = False
    for f in entry.get("features", []):
        if f.get("type") == "Transmembrane":
            if re.search("Helical", f.get("description", "")):
                start_val = f.get("location", {}).get("start", {}).get("value")
                if start_val is not None and int(start_val) <= 90:
                    tm = True
                    break
    return (
        entry["primaryAccession"],
        entry["organism"]["scientificName"],
        get_kingdom(entry),
        entry["sequence"]["length"],
        tm
    )

re_next_link = re.compile(r'<(.+)>; rel="next"')
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

def get_next_link(headers):
    if "Link" in headers:
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)
    return None

def get_batch(batch_url):
    while batch_url:
        response = session.get(batch_url)
        response.raise_for_status()
        total = response.headers.get("x-total-results", "0")
        yield response, total
        batch_url = get_next_link(response.headers)

def get_dataset(search_url, filter_function, json_to_tsv_function, columns, output_file_name, output_fasta_file_name):
    n_total, n_filtered = 0, 0
    with open(output_file_name, "w") as ofs, open(output_fasta_file_name, "w") as ofs_fasta:
        print(*columns, sep="\t", file=ofs)
        for batch, total in get_batch(search_url):
            batch_json = json.loads(batch.text)
            for entry in batch_json.get("results", []):
                n_total += 1
                if filter_function(entry):
                    n_filtered += 1
                    fields = json_to_tsv_function(entry)
                    print(*fields, sep="\t", file=ofs)
                    print(">", entry["primaryAccession"], sep="", file=ofs_fasta)
                    print(entry["sequence"]["value"], file=ofs_fasta)
    print(f"[{output_file_name}] Total retrieved: {n_total:,} | Filtered kept: {n_filtered:,}")

url_positive = "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28fragment%3Afalse%29+AND+%28existence%3A1%29+AND+%28ft_signal_exp%3A*%29%29&size=500"
url_negative = "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28reviewed%3Atrue%29+AND+%28taxonomy_id%3A2759%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28fragment%3Afalse%29+NOT+%28ft_signal%3A*%29+AND+%28+%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0191%29+OR+%28cc_scl_term_exp%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29+OR+%28cc_scl_term_exp%3ASL-0039%29+%29+AND+%28existence%3A1%29%29&size=500"

col_positive = ["Accession", "Organism", "Kingdom", "Sequence length", "SP cleavage"]
col_negative = ["Accession", "Organism", "Kingdom", "Sequence length", "N-term transmembrane"]

print("Downloading Positive Dataset (SP+)...")
get_dataset(url_positive, filter_entry_positive, json_to_tsv_positive, col_positive, "positive.tsv", "positive.fasta")

print("Downloading Negative Dataset (SP-)...")
get_dataset(url_negative, filter_entry_negative, json_to_tsv_negative, col_negative, "negative.tsv", "negative.fasta")

positive_set = pd.read_csv("positive.tsv", sep="\t")
negative_set = pd.read_csv("negative.tsv", sep="\t")
print("\n--- Summary ---")
print(f"Positive sequences: {len(positive_set):,}")
print(f"Negative sequences: {len(negative_set):,}")
print(f"Negative with N-term TM helix: {(negative_set['N-term transmembrane'] == True).sum():,}")

[positive.tsv] Total retrieved: 2,972 | Filtered kept: 2,953
[negative.tsv] Total retrieved: 20,974 | Filtered kept: 20,974

--- Summary ---
Positive sequences: 2,953
Negative sequences: 20,974
Negative with N-term TM helix: 2,531
